# MALLORN Astronomical Classification Challenge

**Goal**: Identify Tidal Disruption Events (TDEs) — stars being torn apart by black holes — from simulated LSST light curves.

**Metric**: F1 Score (binary classification)

**Dataset**: 10,178 simulated LSST light curves constructed from real ZTF observations:
- 64 TDEs (target class)
- 727 Nuclear Supernovae
- 1,407 AGN

**Approach**:
1. Extract statistical and temporal features from multi-band light curves
2. Engineer domain-specific astronomical features (variability indices, color features, rise/decline rates)
3. Train an ensemble of LightGBM + XGBoost + ExtraTrees
4. Optimize threshold for F1 score

## 1. Setup & Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import find_peaks
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")

DATA_DIR = "data"
FILTERS = ["u", "g", "r", "i", "z", "y"]
N_SPLITS = 20

## 2. Load Data

In [ ]:
# Load metadata
train_log = pd.read_csv(os.path.join(DATA_DIR, "train_log.csv"))
test_log = pd.read_csv(os.path.join(DATA_DIR, "test_log.csv"))

print(f"Train: {len(train_log)} objects ({train_log['target'].sum()} TDEs, {len(train_log) - train_log['target'].sum()} non-TDEs)")
print(f"Test:  {len(test_log)} objects")
print(f"\nTDE ratio: {train_log['target'].mean():.2%}")
train_log.head()

In [ ]:
# Load all light curves from 20 splits
train_lcs, test_lcs = [], []
for i in tqdm(range(1, N_SPLITS + 1), desc="Loading splits"):
    d = os.path.join(DATA_DIR, f"split_{i:02d}")
    train_lcs.append(pd.read_csv(os.path.join(d, "train_full_lightcurves.csv")))
    test_lcs.append(pd.read_csv(os.path.join(d, "test_full_lightcurves.csv")))

train_lc = pd.concat(train_lcs, ignore_index=True)
test_lc = pd.concat(test_lcs, ignore_index=True)
del train_lcs, test_lcs

print(f"Train LC rows: {len(train_lc):,}")
print(f"Test  LC rows: {len(test_lc):,}")
train_lc.head()

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
counts = train_log["target"].value_counts().sort_index()
axes[0].bar(["Non-TDE (0)", "TDE (1)"], counts.values, color=["steelblue", "coral"])
axes[0].set_title("Target Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha="center", fontweight="bold")

# SpecType distribution
spec_counts = train_log["SpecType"].value_counts()
colors = ["coral" if s == "TDE" else "steelblue" for s in spec_counts.index]
axes[1].barh(spec_counts.index, spec_counts.values, color=colors)
axes[1].set_title("Spectral Type Distribution")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Redshift distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, "steelblue", "Non-TDE"), (1, "coral", "TDE")]:
    subset = train_log[train_log["target"] == label]
    axes[0].hist(subset["Z"], bins=40, alpha=0.6, label=name, color=color, density=True)
axes[0].set_xlabel("Redshift (Z)")
axes[0].set_ylabel("Density")
axes[0].set_title("Redshift Distribution by Class")
axes[0].legend()

for label, color, name in [(0, "steelblue", "Non-TDE"), (1, "coral", "TDE")]:
    subset = train_log[train_log["target"] == label]
    axes[1].hist(subset["EBV"], bins=40, alpha=0.6, label=name, color=color, density=True)
axes[1].set_xlabel("E(B-V) Extinction")
axes[1].set_ylabel("Density")
axes[1].set_title("Extinction Distribution by Class")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Example light curves: TDE vs Non-TDE
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
filter_colors = {"u": "purple", "g": "green", "r": "red", "i": "orange", "z": "brown", "y": "gray"}

# TDE example
tde_ids = train_log[train_log["target"] == 1]["object_id"].values[:1]
for obj_id in tde_ids:
    obj_lc = train_lc[train_lc["object_id"] == obj_id]
    for filt in FILTERS:
        fdata = obj_lc[obj_lc["Filter"] == filt]
        if len(fdata) > 0:
            axes[0].errorbar(fdata["Time (MJD)"], fdata["Flux"], yerr=fdata["Flux_err"],
                           fmt="o", ms=3, label=filt, color=filter_colors[filt], alpha=0.7)
axes[0].set_title(f"TDE: {tde_ids[0]}")
axes[0].set_xlabel("Time (MJD)")
axes[0].set_ylabel("Flux")
axes[0].legend()

# Non-TDE example (AGN)
agn_ids = train_log[(train_log["target"] == 0) & (train_log["SpecType"] == "AGN")]["object_id"].values[:1]
for obj_id in agn_ids:
    obj_lc = train_lc[train_lc["object_id"] == obj_id]
    for filt in FILTERS:
        fdata = obj_lc[obj_lc["Filter"] == filt]
        if len(fdata) > 0:
            axes[1].errorbar(fdata["Time (MJD)"], fdata["Flux"], yerr=fdata["Flux_err"],
                           fmt="o", ms=3, label=filt, color=filter_colors[filt], alpha=0.7)
axes[1].set_title(f"AGN: {agn_ids[0]}")
axes[1].set_xlabel("Time (MJD)")
axes[1].set_ylabel("Flux")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Feature Engineering

We extract features in several categories:
1. **Global flux statistics** — mean, std, skew, kurtosis, percentiles, MAD
2. **Temporal features** — rise/decline rates, peak timing, time span
3. **Variability indices** — Stetson J, Von Neumann ratio, excess variance
4. **Per-filter features** — statistics and rates for each LSST band (u,g,r,i,z,y)
5. **Color features** — flux differences between bands, peak flux ratios
6. **Cross-filter timing** — peak delay between bands
7. **Metadata interactions** — features combined with redshift and extinction

In [ ]:
def safe_stat(arr, func, default=0):
    try:
        if len(arr) < 2:
            return default
        val = func(arr)
        return val if np.isfinite(val) else default
    except Exception:
        return default


def compute_weighted_mean(flux, flux_err):
    """Inverse-variance weighted mean."""
    w = 1.0 / (flux_err**2 + 1e-10)
    return np.sum(flux * w) / np.sum(w)


def compute_stetson_j(flux, flux_err):
    """Stetson J variability index — detects correlated variability."""
    if len(flux) < 3:
        return 0
    mean = compute_weighted_mean(flux, flux_err)
    residuals = (flux - mean) / (flux_err + 1e-10)
    n = len(flux)
    pairs = residuals[:-1] * residuals[1:]
    sign_pairs = np.sign(pairs)
    return np.sum(sign_pairs * np.sqrt(np.abs(pairs))) / n


def compute_von_neumann_ratio(flux):
    """Von Neumann ratio — measures smoothness of time series."""
    if len(flux) < 3:
        return 0
    var = np.var(flux)
    if var < 1e-10:
        return 0
    delta_sq = np.mean(np.diff(flux)**2)
    return delta_sq / var

In [ ]:
def extract_features_for_object(group):
    """Extract all features from one object's light curve."""
    feats = {}
    flux = group["Flux"].values
    flux_err = group["Flux_err"].values
    time = group["Time (MJD)"].values
    filters = group["Filter"].values

    sort_idx = np.argsort(time)
    flux_s = flux[sort_idx]
    time_s = time[sort_idx]
    flux_err_s = flux_err[sort_idx]
    filters_s = filters[sort_idx]

    n = len(flux)

    # --- Global basic stats ---
    feats["n_obs"] = n
    feats["flux_mean"] = np.mean(flux)
    feats["flux_std"] = np.std(flux) if n > 1 else 0
    feats["flux_median"] = np.median(flux)
    feats["flux_min"] = np.min(flux)
    feats["flux_max"] = np.max(flux)
    feats["flux_range"] = feats["flux_max"] - feats["flux_min"]
    feats["flux_skew"] = safe_stat(flux, stats.skew)
    feats["flux_kurtosis"] = safe_stat(flux, stats.kurtosis)
    feats["flux_iqr"] = np.percentile(flux, 75) - np.percentile(flux, 25)
    feats["flux_p10"] = np.percentile(flux, 10)
    feats["flux_p90"] = np.percentile(flux, 90)
    feats["flux_above_mean_frac"] = np.mean(flux > feats["flux_mean"])
    feats["flux_positive_frac"] = np.mean(flux > 0)
    feats["flux_mad"] = np.median(np.abs(flux - feats["flux_median"]))
    feats["flux_wmean"] = compute_weighted_mean(flux, flux_err)

    # --- Error features ---
    feats["flux_err_mean"] = np.mean(flux_err)
    feats["flux_err_std"] = np.std(flux_err) if n > 1 else 0
    feats["flux_err_median"] = np.median(flux_err)
    feats["snr_mean"] = np.mean(np.abs(flux) / (flux_err + 1e-10))
    feats["snr_max"] = np.max(np.abs(flux) / (flux_err + 1e-10))
    feats["snr_std"] = np.std(np.abs(flux) / (flux_err + 1e-10)) if n > 1 else 0

    # --- Time features ---
    feats["time_span"] = np.ptp(time_s) if n > 1 else 0

    if n > 1:
        dt = np.diff(time_s)
        feats["dt_mean"] = np.mean(dt)
        feats["dt_std"] = np.std(dt)
        feats["dt_min"] = np.min(dt)
        feats["dt_max"] = np.max(dt)
        dflux = np.diff(flux_s)
        rates = dflux / (dt + 1e-10)
        feats["flux_rate_mean"] = np.mean(rates)
        feats["flux_rate_std"] = np.std(rates)
        feats["flux_rate_max"] = np.max(rates)
        feats["flux_rate_min"] = np.min(rates)
        feats["flux_rate_abs_max"] = np.max(np.abs(rates))
        feats["flux_rate_pos_frac"] = np.mean(rates > 0)
    else:
        for k in ["dt_mean", "dt_std", "dt_min", "dt_max",
                   "flux_rate_mean", "flux_rate_std", "flux_rate_max",
                   "flux_rate_min", "flux_rate_abs_max", "flux_rate_pos_frac"]:
            feats[k] = 0

    # --- Peak analysis ---
    peak_idx = np.argmax(flux_s)
    feats["peak_flux"] = flux_s[peak_idx]
    feats["peak_phase"] = (time_s[peak_idx] - time_s[0]) / (feats["time_span"] + 1e-10) if n > 1 else 0.5
    feats["peak_time_from_start"] = time_s[peak_idx] - time_s[0]

    # Rise and decline behavior
    if peak_idx > 0:
        rise_flux = flux_s[:peak_idx + 1]
        rise_time = time_s[:peak_idx + 1]
        feats["rise_rate"] = (rise_flux[-1] - rise_flux[0]) / (rise_time[-1] - rise_time[0] + 1e-10)
        feats["rise_duration"] = rise_time[-1] - rise_time[0]
    else:
        feats["rise_rate"] = 0
        feats["rise_duration"] = 0

    if peak_idx < n - 1:
        dec_flux = flux_s[peak_idx:]
        dec_time = time_s[peak_idx:]
        feats["decline_rate"] = (dec_flux[-1] - dec_flux[0]) / (dec_time[-1] - dec_time[0] + 1e-10)
        feats["decline_duration"] = dec_time[-1] - dec_time[0]
    else:
        feats["decline_rate"] = 0
        feats["decline_duration"] = 0

    feats["rise_decline_ratio"] = feats["rise_duration"] / (feats["decline_duration"] + 1e-10)
    feats["amplitude_snr"] = feats["flux_range"] / (feats["flux_err_mean"] + 1e-10)

    # --- Variability indices ---
    feats["stetson_j"] = compute_stetson_j(flux_s, flux_err_s)
    feats["von_neumann"] = compute_von_neumann_ratio(flux_s)
    mean_err_sq = np.mean(flux_err**2)
    feats["excess_variance"] = (np.var(flux) - mean_err_sq) / (feats["flux_mean"]**2 + 1e-10)

    # Number of peaks
    if n > 5:
        try:
            peaks, _ = find_peaks(flux_s, height=feats["flux_mean"])
            feats["n_peaks"] = len(peaks)
        except Exception:
            feats["n_peaks"] = 0
    else:
        feats["n_peaks"] = 0

    # --- Per-filter features ---
    band_means = {}
    band_peak_flux = {}
    for filt in FILTERS:
        mask = filters == filt
        f_flux = flux[mask]
        f_err = flux_err[mask]
        f_time = time[mask]
        prefix = f"f_{filt}_"
        feats[prefix + "n"] = len(f_flux)

        if len(f_flux) > 0:
            feats[prefix + "mean"] = np.mean(f_flux)
            feats[prefix + "std"] = np.std(f_flux) if len(f_flux) > 1 else 0
            feats[prefix + "max"] = np.max(f_flux)
            feats[prefix + "min"] = np.min(f_flux)
            feats[prefix + "range"] = feats[prefix + "max"] - feats[prefix + "min"]
            feats[prefix + "skew"] = safe_stat(f_flux, stats.skew)
            feats[prefix + "kurtosis"] = safe_stat(f_flux, stats.kurtosis)
            feats[prefix + "snr"] = np.mean(np.abs(f_flux) / (f_err + 1e-10))
            feats[prefix + "wmean"] = compute_weighted_mean(f_flux, f_err)
            feats[prefix + "mad"] = np.median(np.abs(f_flux - np.median(f_flux)))
            feats[prefix + "p10"] = np.percentile(f_flux, 10)
            feats[prefix + "p90"] = np.percentile(f_flux, 90)
            feats[prefix + "positive_frac"] = np.mean(f_flux > 0)
            band_means[filt] = feats[prefix + "mean"]
            band_peak_flux[filt] = feats[prefix + "max"]

            if len(f_flux) > 2:
                si = np.argsort(f_time)
                fs = f_flux[si]
                ts = f_time[si]
                dt_f = np.diff(ts)
                df_f = np.diff(fs)
                rates_f = df_f / (dt_f + 1e-10)
                feats[prefix + "rate_max"] = np.max(np.abs(rates_f))
                feats[prefix + "rate_mean"] = np.mean(rates_f)
                pk = np.argmax(fs)
                feats[prefix + "peak_phase"] = (ts[pk] - ts[0]) / (ts[-1] - ts[0] + 1e-10)
                feats[prefix + "stetson_j"] = compute_stetson_j(f_flux[si], f_err[si])
            else:
                feats[prefix + "rate_max"] = 0
                feats[prefix + "rate_mean"] = 0
                feats[prefix + "peak_phase"] = 0.5
                feats[prefix + "stetson_j"] = 0
        else:
            for sfx in ["mean", "std", "max", "min", "range", "skew", "kurtosis",
                         "snr", "wmean", "mad", "p10", "p90", "positive_frac",
                         "rate_max", "rate_mean", "peak_phase", "stetson_j"]:
                feats[prefix + sfx] = 0
            band_means[filt] = 0
            band_peak_flux[filt] = 0

    # --- Color features ---
    color_pairs = [("u", "g"), ("g", "r"), ("r", "i"), ("i", "z"), ("z", "y"),
                   ("g", "i"), ("u", "r"), ("r", "z")]
    for b1, b2 in color_pairs:
        feats[f"color_{b1}_{b2}"] = band_means[b1] - band_means[b2]
        feats[f"color_peak_{b1}_{b2}"] = band_peak_flux[b1] - band_peak_flux[b2]

    feats["color_g_r_range"] = feats.get("f_g_range", 0) - feats.get("f_r_range", 0)
    feats["color_r_i_range"] = feats.get("f_r_range", 0) - feats.get("f_i_range", 0)

    # --- Cross-filter peak timing ---
    peak_times = {}
    for filt in FILTERS:
        mask = filters == filt
        f_flux = flux[mask]
        f_time = time[mask]
        peak_times[filt] = f_time[np.argmax(f_flux)] if len(f_flux) > 0 else np.nan

    for b1, b2 in [("g", "r"), ("r", "i"), ("i", "z")]:
        if not np.isnan(peak_times.get(b1, np.nan)) and not np.isnan(peak_times.get(b2, np.nan)):
            feats[f"peak_delay_{b1}_{b2}"] = peak_times[b1] - peak_times[b2]
        else:
            feats[f"peak_delay_{b1}_{b2}"] = 0

    return feats

In [ ]:
def build_features(lc_df, meta_df, desc="Features"):
    """Build full feature matrix from light curves + metadata."""
    grouped = lc_df.groupby("object_id")
    records = []
    for obj_id, group in tqdm(grouped, desc=desc):
        feats = extract_features_for_object(group)
        feats["object_id"] = obj_id
        records.append(feats)

    feat_df = pd.DataFrame(records)
    merged = meta_df.merge(feat_df, on="object_id", how="left")

    merged["Z"] = pd.to_numeric(merged["Z"], errors="coerce")
    merged["Z_err"] = pd.to_numeric(merged["Z_err"], errors="coerce")
    merged["EBV"] = pd.to_numeric(merged["EBV"], errors="coerce")

    # Interaction features with redshift
    merged["flux_mean_over_z"] = merged["flux_mean"] / (merged["Z"] + 1e-10)
    merged["flux_range_over_z"] = merged["flux_range"] / (merged["Z"] + 1e-10)
    merged["amplitude_over_z"] = merged["amplitude_snr"] / (merged["Z"] + 1e-10)
    merged["peak_flux_over_z"] = merged["peak_flux"] / (merged["Z"] + 1e-10)
    merged["Z_times_EBV"] = merged["Z"] * merged["EBV"]
    merged["log_flux_range"] = np.log1p(np.abs(merged["flux_range"]))
    merged["log_peak_flux"] = np.log1p(np.abs(merged["peak_flux"]))
    merged["has_z_err"] = (~merged["Z_err"].isna()).astype(int)

    return merged

In [ ]:
print("Building train features...")
train_feat = build_features(train_lc, train_log, desc="Train features")
print(f"\nBuilding test features...")
test_feat = build_features(test_lc, test_log, desc="Test features")

print(f"\nTrain shape: {train_feat.shape}")
print(f"Test shape:  {test_feat.shape}")

## 5. Model Training

We use a 3-model ensemble:
- **LightGBM** — fast gradient boosting, good with tabular data
- **XGBoost** — robust gradient boosting with regularization
- **ExtraTrees** — randomized tree ensemble for diversity

All models use `scale_pos_weight` or `class_weight="balanced"` to handle the severe class imbalance (~5% TDE).

In [ ]:
# Prepare data
exclude = {"object_id", "SpecType", "English Translation", "split", "target"}
feature_cols = [c for c in train_feat.columns if c not in exclude
                and train_feat[c].dtype in [np.float64, np.int64, float, int]]

X = train_feat[feature_cols].values.astype(np.float32)
y = train_feat["target"].values.astype(int)
X_test = test_feat[feature_cols].values.astype(np.float32)

X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

neg, pos = (y == 0).sum(), (y == 1).sum()
scale_pos = neg / max(pos, 1)

print(f"Features: {len(feature_cols)}")
print(f"Class ratio: {neg}:{pos} (scale_pos_weight={scale_pos:.1f})")

In [ ]:
# Model hyperparameters
lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.02,
    "num_leaves": 31,
    "max_depth": 7,
    "min_child_samples": 3,
    "scale_pos_weight": scale_pos,
    "subsample": 0.75,
    "colsample_bytree": 0.75,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42,
}

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "learning_rate": 0.02,
    "max_depth": 7,
    "min_child_weight": 3,
    "scale_pos_weight": scale_pos,
    "subsample": 0.75,
    "colsample_bytree": 0.75,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "verbosity": 0,
    "nthread": -1,
    "random_state": 42,
}

In [ ]:
# 5-Fold Cross Validation with Ensemble
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(y))
oof_xgb = np.zeros(len(y))
oof_et = np.zeros(len(y))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_et = np.zeros(len(X_test))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{n_folds} ---")
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    print(f"  Train: {len(y_tr)} ({y_tr.sum()} TDE) | Val: {len(y_val)} ({y_val.sum()} TDE)")

    # LightGBM
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_val_ds = lgb.Dataset(X_val, y_val, reference=lgb_train)
    lgb_model = lgb.train(
        lgb_params, lgb_train, num_boost_round=3000,
        valid_sets=[lgb_val_ds],
        callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)],
    )
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    test_lgb += lgb_model.predict(X_test) / n_folds

    # XGBoost
    xgb_tr = xgb.DMatrix(X_tr, label=y_tr)
    xgb_vl = xgb.DMatrix(X_val, label=y_val)
    xgb_model = xgb.train(
        xgb_params, xgb_tr, num_boost_round=3000,
        evals=[(xgb_vl, "val")], early_stopping_rounds=150, verbose_eval=False,
    )
    oof_xgb[val_idx] = xgb_model.predict(xgb.DMatrix(X_val))
    test_xgb += xgb_model.predict(xgb.DMatrix(X_test)) / n_folds

    # Extra Trees
    et = ExtraTreesClassifier(
        n_estimators=500, max_depth=10, min_samples_leaf=3,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )
    et.fit(X_tr, y_tr)
    oof_et[val_idx] = et.predict_proba(X_val)[:, 1]
    test_et += et.predict_proba(X_test)[:, 1] / n_folds

    # Fold score (ensemble)
    oof_ens = 0.4 * oof_lgb[val_idx] + 0.4 * oof_xgb[val_idx] + 0.2 * oof_et[val_idx]
    best_f1 = max(
        f1_score(y_val, (oof_ens >= t).astype(int))
        for t in np.arange(0.05, 0.9, 0.01)
    )
    fold_scores.append(best_f1)
    print(f"  Fold F1: {best_f1:.4f}")

print(f"\nCV Mean F1: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")

## 6. Ensemble Weight & Threshold Optimization

In [ ]:
# Search best ensemble weights on OOF predictions
best_f1_global = 0
best_w = (0.4, 0.4, 0.2)
best_thr = 0.5

for w_lgb in np.arange(0.2, 0.7, 0.1):
    for w_xgb in np.arange(0.2, 0.7, 0.1):
        w_et = 1.0 - w_lgb - w_xgb
        if w_et < 0.05 or w_et > 0.5:
            continue
        oof_ens = w_lgb * oof_lgb + w_xgb * oof_xgb + w_et * oof_et
        for thr in np.arange(0.05, 0.8, 0.005):
            f1 = f1_score(y, (oof_ens >= thr).astype(int))
            if f1 > best_f1_global:
                best_f1_global = f1
                best_w = (w_lgb, w_xgb, w_et)
                best_thr = thr

print(f"Best ensemble weights: LGB={best_w[0]:.1f}, XGB={best_w[1]:.1f}, ET={best_w[2]:.1f}")
print(f"Best OOF F1: {best_f1_global:.4f} at threshold={best_thr:.3f}")

In [ ]:
# Classification report & confusion matrix on OOF
oof_final = best_w[0] * oof_lgb + best_w[1] * oof_xgb + best_w[2] * oof_et
oof_pred = (oof_final >= best_thr).astype(int)

print(classification_report(y, oof_pred, target_names=["Non-TDE", "TDE"]))

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y, oof_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Non-TDE", "TDE"])
disp.plot(ax=ax, cmap="Blues")
ax.set_title(f"OOF Confusion Matrix (F1={best_f1_global:.4f})")
plt.tight_layout()
plt.show()

In [ ]:
# F1 vs threshold curve
thresholds = np.arange(0.05, 0.8, 0.005)
f1_scores_curve = [f1_score(y, (oof_final >= t).astype(int)) for t in thresholds]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, f1_scores_curve, linewidth=2)
plt.axvline(best_thr, color="red", linestyle="--", label=f"Best threshold={best_thr:.3f}")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score vs Classification Threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
# Train final LightGBM on all data for feature importance
lgb_full = lgb.LGBMClassifier(**{**lgb_params, "n_estimators": 1000})
lgb_full.fit(X, y)

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": lgb_full.feature_importances_
}).sort_values("importance", ascending=False)

# Plot top 25 features
top_n = 25
top_feats = importance.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_feats["importance"].values[::-1], color="steelblue")
plt.yticks(range(top_n), top_feats["feature"].values[::-1])
plt.xlabel("Importance (split count)")
plt.title(f"Top {top_n} Feature Importances (LightGBM)")
plt.tight_layout()
plt.show()

## 8. Generate Submission

In [ ]:
# Final prediction on test set
test_preds = best_w[0] * test_lgb + best_w[1] * test_xgb + best_w[2] * test_et

submission = pd.DataFrame({
    "object_id": test_log["object_id"],
    "prediction": (test_preds >= best_thr).astype(int),
})

# Sanity checks
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
assert list(submission.columns) == list(sample.columns), "Column mismatch!"
assert len(submission) == len(sample), "Row count mismatch!"

print(f"Submission shape: {submission.shape}")
print(f"Predicted TDEs: {submission['prediction'].sum()} / {len(submission)} ({submission['prediction'].mean():.2%})")
print(f"\nPrediction distribution:")
print(submission["prediction"].value_counts())

submission.to_csv("submission.csv", index=False)
print("\nSaved to submission.csv")
submission.head(10)